In [1]:
!pip install mlflow boto3 awscli optuna xgboost imbalanced-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.1/51.1 kB 5.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 108.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 118.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 89.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.2/20.2 MB 95.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.1/16.1 MB 102.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:

from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTEENN
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os


In [3]:
import mlflow
# Step 2: Set up the MLflow tracking server
mlflow.set_tracking_uri("http://ec2-54-179-95-179.ap-southeast-1.compute.amazonaws.com:5000/")

In [4]:
df = pd.read_csv('/content/data_prerpocessing.csv').dropna(subset=['review'])
df.shape

(9942, 2)

In [5]:
# Set or create an experiment
mlflow.set_experiment("Exp 5 - ML Algos with HP Tuning")

<Experiment: artifact_location='mlflow-artifacts:/5', creation_time=1791949221161, effective_trace_archival_retention=None, experiment_id='5', last_update_time=1791949221161, lifecycle_stage='active', name='Exp 5 - ML Algos with HP Tuning', tags={}, trace_location=None, workspace='default'>

In [6]:
import mlflow
TRACKING_URI = (
    "http://ec2-54-179-95-179.ap-southeast-1.compute.amazonaws.com:5000"
)

mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment("Shopee_Sentiment_ArtifactProxy_v2")

exp = mlflow.get_experiment_by_name(
    "Shopee_Sentiment_ArtifactProxy_v2"
)

print("Tracking URI:", mlflow.get_tracking_uri())
print("Experiment ID:", exp.experiment_id)
print("Artifact location:", exp.artifact_location)

Tracking URI: http://ec2-54-179-95-179.ap-southeast-1.compute.amazonaws.com:5000
Experiment ID: 3
Artifact location: mlflow-artifacts:/3


In [7]:

import optuna
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

In [8]:
import mlflow
import mlflow.sklearn
import optuna
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    balanced_accuracy_score
)
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

# ============================================================
# Step 1: Clean data and validate binary labels
# ============================================================

df = df.dropna(subset=["review", "label"]).copy()

df["review"] = df["review"].astype(str).str.strip()
df = df[df["review"] != ""]

# Normalize labels
df["label"] = df["label"].astype(str).str.strip().str.lower()

# Keep only positive and negative
df = df[df["label"].isin(["positive", "negative"])].copy()

# Map labels to binary values
# negative = 0, positive = 1
df["label"] = df["label"].map({
    "negative": 0,
    "positive": 1
}).astype(int)

print("Label distribution:")
print(df["label"].value_counts())
print("\nLabel ratio:")
print(df["label"].value_counts(normalize=True))

# ============================================================
# Step 2: Split data before vectorization and resampling
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    df["review"],
    df["label"],
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

# ============================================================
# Step 3: TF-IDF vectorization
# ============================================================

ngram_range = (1, 3)
max_features = 10000

vectorizer = TfidfVectorizer(
    ngram_range=ngram_range,
    max_features=max_features,
    sublinear_tf=True
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# ============================================================
# Step 4: Apply SMOTE to training data only
# ============================================================

smote = SMOTE(random_state=42)

X_train_resampled, y_train_resampled = smote.fit_resample(
    X_train_vec,
    y_train
)

print("\nTraining samples before SMOTE:", len(y_train))
print("Training samples after SMOTE:", len(y_train_resampled))
print("\nResampled label distribution:")
print(y_train_resampled.value_counts())

# ============================================================
# Step 5: Configure MLflow
# ============================================================

TRACKING_URI = (
    "http://ec2-54-179-95-179.ap-southeast-1.compute.amazonaws.com:5000"
)

mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment("Shopee_Sentiment_ArtifactProxy_v2")

# ============================================================
# Step 6: Optuna objective using cross-validation
# ============================================================

def objective_xgboost(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 300),
        "learning_rate": trial.suggest_float(
            "learning_rate", 1e-3, 0.1, log=True
        ),
        "max_depth": trial.suggest_int("max_depth", 3, 8),
        "subsample": trial.suggest_float("subsample", 0.7, 1.0),
        "colsample_bytree": trial.suggest_float(
            "colsample_bytree", 0.7, 1.0
        ),
        "objective": "binary:logistic",
        "eval_metric": "logloss",
        "random_state": 42,
        "n_jobs": -1
    }

    model = XGBClassifier(**params)

    # Use cross-validation on training data only.
    # Do not use the test set to select hyperparameters.
    cv = StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=42
    )

    scores = cross_val_score(
        model,
        X_train_resampled,
        y_train_resampled,
        cv=cv,
        scoring="accuracy",
        n_jobs=1
    )

    return float(np.mean(scores))


# ============================================================
# Step 7: Tune XGBoost
# ============================================================

study = optuna.create_study(direction="maximize")

study.optimize(
    objective_xgboost,
    n_trials=30
)

print("\nBest CV accuracy:", study.best_value)
print("Best parameters:", study.best_params)

# ============================================================
# Step 8: Train final model using the best parameters
# ============================================================

best_model = XGBClassifier(
    **study.best_params,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

best_model.fit(
    X_train_resampled,
    y_train_resampled
)

# ============================================================
# Step 9: Evaluate once on untouched test set
# ============================================================

y_pred = best_model.predict(X_test_vec)

accuracy = accuracy_score(y_test, y_pred)
balanced_acc = balanced_accuracy_score(y_test, y_pred)

print("\nTest Accuracy:", accuracy)
print("Test Balanced Accuracy:", balanced_acc)

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    labels=[0, 1],
    target_names=["negative", "positive"],
    zero_division=0
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred, labels=[0, 1]))

# ============================================================
# Step 10: Log final experiment to MLflow
# ============================================================

with mlflow.start_run(run_name="XGBoost_SMOTE_TFIDF_Trigrams"):

    mlflow.set_tag("experiment_type", "algorithm_comparison")
    mlflow.set_tag("model_type", "XGBoost")
    mlflow.set_tag("vectorizer_type", "TF-IDF")
    mlflow.set_tag("imbalance_method", "SMOTE")

    mlflow.log_param("ngram_range", str(ngram_range))
    mlflow.log_param("max_features", max_features)
    mlflow.log_param("smote_random_state", 42)
    mlflow.log_param("best_cv_accuracy", study.best_value)

    for param, value in study.best_params.items():
        mlflow.log_param(param, value)

    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("balanced_accuracy", balanced_acc)

    report = classification_report(
        y_test,
        y_pred,
        labels=[0, 1],
        target_names=["negative", "positive"],
        output_dict=True,
        zero_division=0
    )

    for label_name, metrics in report.items():
        if isinstance(metrics, dict):
            for metric_name, value in metrics.items():
                mlflow.log_metric(
                    f"{label_name}_{metric_name}",
                    value
                )

    # Log XGBoost model
    mlflow.xgboost.log_model(
        xgb_model=best_model,
        name="xgboost_tfidf_trigrams_smote"
    )

    # Save the fitted vectorizer for future inference
    import joblib

    joblib.dump(vectorizer, "tfidf_vectorizer.pkl")
    mlflow.log_artifact("tfidf_vectorizer.pkl")

    print("\nMLflow run ID:", mlflow.active_run().info.run_id)

print("\nExperiment completed.")


Label distribution:
label
1    6259
0    3683
Name: count, dtype: int64

Label ratio:
label
1    0.629551
0    0.370449
Name: proportion, dtype: float64

Training samples before SMOTE: 7953
Training samples after SMOTE: 10014

Resampled label distribution:
label
1    5007
0    5007
Name: count, dtype: int64


[I 2026-10-14 11:37:09,023] A new study created in memory with name: no-name-80cf065a-7086-4f7f-aede-ccead3f68b83
[I 2026-10-14 11:37:36,779] Trial 0 finished with value: 0.8653884561613742 and parameters: {'n_estimators': 157, 'learning_rate': 0.03582360127011488, 'max_depth': 4, 'subsample': 0.9687688703234738, 'colsample_bytree': 0.8247482586634108}. Best is trial 0 with value: 0.8653884561613742.
[I 2026-10-14 11:39:22,046] Trial 1 finished with value: 0.895945676053525 and parameters: {'n_estimators': 290, 'learning_rate': 0.04675442866992319, 'max_depth': 7, 'subsample': 0.9454604031466681, 'colsample_bytree': 0.7992746683135938}. Best is trial 1 with value: 0.895945676053525.
[I 2026-10-14 11:39:38,605] Trial 2 finished with value: 0.7986818454164171 and parameters: {'n_estimators': 144, 'learning_rate': 0.003295151884261112, 'max_depth': 3, 'subsample': 0.8631050351608331, 'colsample_bytree': 0.8620448294656128}. Best is trial 1 with value: 0.895945676053525.
[I 2026-10-14 11:4


Best CV accuracy: 0.9024365887757141
Best parameters: {'n_estimators': 293, 'learning_rate': 0.09428383437555016, 'max_depth': 8, 'subsample': 0.9389019716578857, 'colsample_bytree': 0.8050765065367762}

Test Accuracy: 0.8768225238813474
Test Balanced Accuracy: 0.8675524859004426

Classification Report:
              precision    recall  f1-score   support

    negative       0.84      0.83      0.83       737
    positive       0.90      0.90      0.90      1252

    accuracy                           0.88      1989
   macro avg       0.87      0.87      0.87      1989
weighted avg       0.88      0.88      0.88      1989


Confusion Matrix:
[[ 613  124]
 [ 121 1131]]

MLflow run ID: d5286020a4ef43c5b31c70b2b1b528b7
🏃 View run XGBoost_SMOTE_TFIDF_Trigrams at: http://ec2-54-179-95-179.ap-southeast-1.compute.amazonaws.com:5000/#/experiments/3/runs/d5286020a4ef43c5b31c70b2b1b528b7
🧪 View experiment at: http://ec2-54-179-95-179.ap-southeast-1.compute.amazonaws.com:5000/#/experiments/3

E